# Modelo pedidos digitales

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import col, lit, udf
from pyspark.sql.window import Window

In [0]:
spark = SparkSession.builder.appName("Modelo Pedidos Digitales").getOrCreate()

In [0]:
df = spark.read.table("workspace.default.transaccional_clientes")
display(df.limit(10))

In [0]:
id_cols = ['cliente_id', 'agencia_id', 'ruta_id']

cat_cols = ['pais', 'region_comercial', 'tipo_cliente', 'madurez_digital', 'frecuencia_visitas', 'canal_pedido']

disc_cols = ['estrellas', 'materiales_distintos']

num_cols = ['facturacion_usd', 'cajas_fisicas']

date_cols = ['fecha_pedido_dt']

## Distribuciones

In [0]:
for col_name in num_cols:
    data = df.select(col_name).dropna().toPandas()
    plt.figure(figsize=(8, 4))
    sns.histplot(data[col_name], kde=True, bins=30)
    plt.title(f'Distribución de {col_name}')
    plt.xlabel(col_name)
    plt.ylabel('Frecuencia')
    plt.show()

In [0]:
for col_name in disc_cols:
    total_count = df.filter(col(col_name).isNotNull()).count()
    count_df = (
        df.groupBy(col_name)
        .agg(f.count("*").alias("count"))
        .withColumn("percentage", col("count") / total_count)
        .orderBy(col(col_name).asc())
    )
    display(count_df)

In [0]:
for col_name in cat_cols:
    total_count = df.filter(col(col_name).isNotNull()).count()
    count_df = (
        df.groupBy(col_name)
        .agg(f.count("*").alias("count"))
        .withColumn("percentage", col("count") / total_count)
        .orderBy(col("count").desc())
    )
    display(count_df)

## Análisis de Datos

In [0]:
cliente_canal = (
    df
    .groupBy("cliente_id", "canal_pedido")
    .agg(f.count("*").alias("count"))
)

total_por_cliente = df.groupBy("cliente_id").agg(f.count("*").alias("total"))

cliente_canal = (
    cliente_canal
    .join(total_por_cliente, on="cliente_id", how="left")
    .withColumn("percentage", col("count") / col("total"))
    .select("cliente_id", "canal_pedido", "count", "percentage")
)

# Convert to pandas for plotting
cliente_canal_pd = cliente_canal.toPandas()

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=cliente_canal_pd,
    x="percentage",
    hue="canal_pedido"
)
plt.title("Histograma del porcentaje por canal_pedido")
plt.xlabel("Porcentaje")
plt.ylabel("Frecuencia")
plt.legend(title="Canal Pedido")
plt.show()

In [0]:
df_ym = df.withColumn(
    "year_month",
    f.date_format(col("fecha_pedido"), "yyyyMM").cast("int")
)

cliente_min_max = (
    df_ym.groupBy("cliente_id")
    .agg(
        f.min("year_month").alias("min_year_month"),
        f.max("year_month").alias("max_year_month"),
        f.countDistinct("year_month").alias("meses_activo")
    )
)

cliente_min_max = (
    cliente_min_max
    .withColumn("min_date", f.to_date(f.concat_ws("-", col("min_year_month").cast("string").substr(1,4), col("min_year_month").cast("string").substr(5,2), f.lit("01"))))
    .withColumn("max_date", f.to_date(f.concat_ws("-", col("max_year_month").cast("string").substr(1,4), col("max_year_month").cast("string").substr(5,2), f.lit("01"))))
    .withColumn("meses_total", f.months_between(col("max_date"), col("min_date")) + 1)
    .withColumn("meses_total", col("meses_total").cast("int"))
    .select("cliente_id", "min_year_month", "max_year_month", "meses_total", "meses_activo")
)

display(cliente_min_max.limit(20))

## Creación del dataset

Creamos nuestro target como el canal del siguiente pedido. Eliminamos los s ultimos pedidos de cada cliente, al no saber cual será su siguiente canal.

In [0]:
window_cliente = Window.partitionBy("cliente_id").orderBy("fecha_pedido")

df_target = df.withColumn(
    "canal_target",
    f.lead("canal_pedido").over(window_cliente)
).filter(col("canal_target").isNotNull())

print('Dataset original:', df.count())
print('Dataset con target:', df_target.count())

display(df_target.limit(10))